In [1]:
import os
from collections import defaultdict
import json

import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm import tqdm
tqdm.pandas()

In [8]:
FISCAL_START = 2019
FISCAL_END = 2020

in_dir = f'../data/metro_region_geohash_stops_{FISCAL_START}_{FISCAL_END}'

metro_minmax = defaultdict(list)
for file_name in tqdm(os.listdir(in_dir)):
    metro_name = file_name.replace('.geojson', '') # extract just the metro name
    file_path = os.path.join(in_dir, file_name)
    # print(file_path)
    
    gdf = gpd.read_file(file_path)

    # Some files may not include `prop_subset_stops` (e.g., if the input data was missing)
    # so we fall back to computing it from `total_stops` if available.
    if 'prop_subset_stops' in gdf.columns:
        prop_series = gdf['prop_subset_stops']
    elif 'total_stops' in gdf.columns:
        total_stops = gdf['total_stops'].sum()
        if total_stops == 0:
            prop_series = gdf['total_stops'] * 0.0
        else:
            prop_series = gdf['total_stops'] / total_stops
    elif 'prop_total_stops' in gdf.columns:
        prop_series = gdf['prop_total_stops']
    else:
        print(f"Warning: {metro_name} has no 'prop_subset_stops' or 'total_stops' columns; skipping")
        continue

    if prop_series.empty:
        min_val = max_val = None
    else:
        min_val = prop_series.min()
        max_val = prop_series.max()

    metro_minmax[metro_name].extend([min_val, max_val])


  7%|▋         | 20/300 [00:24<05:56,  1.27s/it]

  8%|▊         | 24/300 [00:28<04:18,  1.07it/s]

 14%|█▎        | 41/300 [00:41<02:31,  1.70it/s]

 26%|██▌       | 78/300 [01:37<04:18,  1.17s/it]

 28%|██▊       | 83/300 [01:39<02:11,  1.65it/s]

...

 86%|████████▌ | 258/300 [04:50<00:37,  1.12it/s]

 89%|████████▉ | 268/300 [05:03<00:39,  1.22s/it]

 90%|█████████ | 271/300 [05:04<00:18,  1.61it/s]

 97%|█████████▋| 292/300 [05:28<00:08,  1.07s/it]

100%|██████████| 300/300 [05:34<00:00,  1.11s/it]


### Diagnostic: OR metro geohash coverage in the activity file
This section checks whether the Oregon metros in the geohash mapping actually have any matching geohashes in the activity dataset. If the match count is zero, that means the activity source contains no stops for those geohashes (so they end up empty).

In [ ]:
# # Diagnose why some OR metros (e.g., Bend, Salem) produce empty outputs
# # by checking whether their geohashes appear in the activity file.

# df_geo = pd.read_csv(f'../data/metro_region_geohashes_no_military.csv')
# act = pd.read_csv(f'../data/activity_{FISCAL_START}-04-01_{FISCAL_END}-03-31.csv')
# act.columns = act.columns.str.lower()

# geo_col = 'geohash6' if 'geohash6' in act.columns else ('geohash' if 'geohash' in act.columns else None)
# if geo_col is None:
#     print('No geohash column found in activity file (expected geohash6 or geohash)')
# else:
#     act_geos = set(act[geo_col].astype(str).unique())
#     or_metros = sorted(df_geo[df_geo['name'].str.endswith(', OR')]['name'].unique())
#     print('OR metros in mapping:', or_metros)
#     for metro in or_metros:
#         metro_geos = set(df_geo.loc[df_geo['name'] == metro, 'geohash'].astype(str))
#         matched = len(metro_geos & act_geos)
#         print(f"{metro}: {matched} / {len(metro_geos)} geohashes found in activity file")

CA metros in mapping: ['Bakersfield, CA', 'Chico, CA', 'El Centro, CA', 'Fresno, CA', 'Los Angeles-Long Beach-Anaheim, CA', 'Merced, CA', 'Modesto, CA', 'Oxnard-Thousand Oaks-Ventura, CA', 'Redding, CA', 'Riverside-San Bernardino-Ontario, CA', 'Sacramento-Roseville-Folsom, CA', 'Salinas, CA', 'San Diego-Chula Vista-Carlsbad, CA', 'San Francisco-Oakland-Berkeley, CA', 'San Jose-Sunnyvale-Santa Clara, CA', 'San Luis Obispo-Paso Robles, CA', 'Santa Cruz-Watsonville, CA', 'Santa Maria-Santa Barbara, CA', 'Santa Rosa-Petaluma, CA', 'Stockton, CA', 'Vallejo, CA', 'Visalia, CA', 'Yuba City, CA']
Bakersfield, CA: 20428 / 32794 geohashes found in activity file
Chico, CA: 5711 / 7765 geohashes found in activity file
El Centro, CA: 8177 / 16071 geohashes found in activity file
Fresno, CA: 15141 / 26464 geohashes found in activity file
Los Angeles-Long Beach-Anaheim, CA: 17339 / 20076 geohashes found in activity file
Merced, CA: 6450 / 8718 geohashes found in activity file
Modesto, CA: 4803 / 6692

In [9]:
# Convert defaultdict to a regular dict
metro_dict = dict(metro_minmax)

# Save to a JSON file
output_path = f'../src/data/min_max_{FISCAL_START}_{FISCAL_END}.json'
with open(output_path, 'w') as json_file:
    json.dump(metro_dict, json_file, indent=2)

print(f"Data has been saved to {output_path}")

Data has been saved to ../src/data/min_max_2019_2020.json
